In [1]:
from pathlib import Path

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from core import Config
import pandas as pd

config = Config()
DATASET: str = '2025basis_thresh_50_win_log-r'
DATA_PATH: Path = config.training_dir / f'{DATASET}.csv'
# variables
DEPTH: int = 2
ITERATIONS: int = 200
LEARNING_RATE: float = 0.1
FOLDS: int = 5
#ORDERED: str | None = None
ORDERED: str | None = "Ordered"
SHUFFLE: bool = True
STATE: int = 42
RESULTS_NAME: str = f'{DEPTH}_{DATASET}'
if ORDERED:
    RESULTS_NAME += f'_{ORDERED}'
if SHUFFLE:
    RESULTS_NAME += f'_sh{STATE}'
SHAP_NAME: str = f'{RESULTS_NAME}_shap_importance.csv'
SHAP_PLOT_NAME: str = f'{RESULTS_NAME}_shap_beeswarm.png'

MODEL_NAME: str = f'4_basis_thresh_50_win_log-r_Ordered_sh42_model.bin'

all_data: pd.DataFrame = pd.read_csv(DATA_PATH, index_col=0)

In [2]:
old_mask = all_data["Date"] < 2025
test_mask = all_data["Date"] == 2025

trainval_data = all_data.loc[old_mask].copy()
test_data = all_data.loc[test_mask].copy()

# date could also be a categorical variable
cat_cols: list[str] = ["Instrument", "TR.GICSSectorCode", "TR.HQCountryCode"]
cat_cols.remove("Instrument")
# disable categorical features when using one-hot encoded datasets
#cat_cols: list[str] = []

In [3]:
# ---- Final evaluation on 2025 data ----
from catboost import Pool, CatBoostRegressor
from typing import cast

# same target and feature processing as for training_data
test_data = test_data.copy()
test_data.drop(
    test_data[test_data["TR.UpstreamScope3PurchasedGoodsAndServices"].isna()].index,
    inplace=True,
)

y_test = test_data["TR.UpstreamScope3PurchasedGoodsAndServices"].to_frame()
X_test = test_data.drop("TR.UpstreamScope3PurchasedGoodsAndServices", axis=1)

X_test[cat_cols] = X_test[cat_cols].astype("str")

test_pool = Pool(
    data=X_test.drop(columns=["Instrument"]),  # mirroring your X_without_instrument
    label=y_test.values.ravel(),
    cat_features=cat_cols,
)

loaded_model = CatBoostRegressor()
loaded_model.load_model(f'{config.results_dir}/model/{MODEL_NAME}')

y_test_pred = loaded_model.predict(test_pool)

rmse_test = float(np.sqrt(mean_squared_error(y_test, y_test_pred)))
mae_test = cast(float, mean_absolute_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

print("2025 test RMSE:", rmse_test)
print("2025 test MAE:", mae_test)
print("2025 test R2:", r2_test)

2025 test RMSE: 1.3703198740838065
2025 test MAE: 0.9518242540114747
2025 test R2: 0.7683066142577535
